## Assignment 06: Data Science II

### Setup

In [ ]:
!pip install scikit-learn
!pip install matplotlib
!pip install pandas

In [ ]:
from matplotlib import pyplot as plt

def plot_df_scatter(dataframe, column_x, column_y, colors=None):
    if colors is not None:
        plt.scatter(dataframe[column_x], dataframe[column_y], s=2, c=colors)
    else:
        plt.scatter(dataframe[column_x], dataframe[column_y], s=2)
    plt.xlabel(column_x)
    plt.ylabel(column_y)

In [ ]:
import pandas as pd
df = pd.read_csv("data/machine_data.csv")
df = df.drop(["Timestamp", "Machine"], axis="columns")

plot_df_scatter(df, "Power", "Speed")

### Task 1: Clustering with K-Means

In the setup code above, data is read from the file ``data/machine_data.csv``.

The file contains operating data from various machines that were recorded during production.
Each entry consists of a time stamp, the machine number and the power consumption ("Power") measured on the machine at that time in kW and the current speed ("Speed") in thousands of revolutions per minute of the machine.

The aim of this task is to reconstruct the recording machine as precisely as possible using the measured values, i.e. to cluster the data points to the machine numbers (types).
The clustering is to be carried out using the "Power" and "Speed" features, which is why the "Timestamp" columns and the "Machine" column to be reconstructed are initially removed from the data in the code above.

A visualization function ``plot_df_scatter`` has already been defined in the setup code and the function is used to visualize the data set.

For the human observer, it seems intuitively easy to cluster the data points.
What is not quite so obvious, however, is what exactly an associated algorithm for automated clustering looks like.

In the following, we will implement the *K-Means* algorithm presented in the lecture. 

As a reminder:

<img style="padding-left:25%; padding-right:25%" width=50% src="figures/K-Means_en.png"/>

#### 1 A) Distance function

We start by implementing a suitable distance function.
This is required in step 2) of the algorithm in order to be able to determine the distance to the respective cluster centers for each of the points.

We choose the Euclidean norm in 2D space as the distance function. For two points $(x_1, y_1), (x_2, y_2) \in \mathbb{R}^2$, this distance can be calculated using the following formula:

$|(x_1, y_1) - (x_2, y_2)|_2 = \sqrt{|x_1-x_2|^2 + |y_1-y_2|^2}$

Implement the distance function ``euklidean_distance_2d(x_1, y_1, x_2, y_2)`` so that the Euclidean distance between the two points is returned.

In [ ]:
import math

def euklidean_distance_2d(x_1, y_1, x_2, y_2):
    ### BEGIN SOLUTION
    dist_squared = (x_1 - x_2)**2 + (y_1 - y_2)**2
    return math.sqrt(dist_squared)
    ### END SOLUTION

In [ ]:
assert euklidean_distance_2d(0, 0, 0, 0) == 0
assert euklidean_distance_2d(1, 0, 0, 1) == euklidean_distance_2d(0, -1, 1, 0)
assert math.isclose(euklidean_distance_2d(1.3, -1.5, -0.5, -2.2), 1.93132, abs_tol=1e-4)

#### 1 B) Cluster assignment

Now we complete the implementation of step 2 of the *K-Means* algorithm.
To do this, we implement the functions ``assign_to_cluster(x, y, cluster_indices, cluster_centers)`` and ``assign_clusters(data_x, data_y, cluster_indices, cluster_centers)``.

*Note:* The following parameter names and descriptions are relevant for the entire exercise sheet and are also used equivalently in the following subtasks:
* ``cluster_indices``: A duplicate-free list with integer indices (identifiers) of the clusters to be optimized, e.g. ``[1,2,3]``. (The exact values are not relevant).
* ``cluster_centers``: A list of the cluster centers (focal points). The length of the list corresponds to the length of the ``cluster_indices`` list. Each entry is a tuple ``(x, y)``, which specifies the x- and y-coordinate of the cluster center.
* ``data_x``: A list of the x-coordinates of all data points.
* ``data_y``: A list of the y-coordinates of all data points.

Proceed as follows to implement the two functions:

1. the function ``assign_to_cluster(x, y, cluster_indices, cluster_centers)`` receives as argument the x- and y-coordinate of any data point, as well as the list of all cluster centers ``cluster_centers``.
The function should return the index of the cluster (according to the list ``cluster_indices``) with minimum distance to the present data point (``x``, ``y``).
Use the previously implemented function ``euklidean_distance_2d`` to calculate the distance.

2. the function ``assign_clusters(data_x, data_y, cluster_indices, cluster_centers)`` contains in the parameters ``data_x`` and ``data_y`` the list of x- and y-coordinates of all data points of the data set.
Implement the function so that a list ``cluster_assignments`` is returned which contains the index of the nearest cluster center for each point of the data set. (The ``i``-th entry of the list should contain the index of the next cluster center for the data point ``data_x[i], data_y[i]``).

In [ ]:
def assign_to_cluster(x, y, cluster_indices, cluster_centers):
    min_index = -1
    min_distance = 1e6
    for i, cluster_index in enumerate(cluster_indices):
        center_x, center_y = cluster_centers[i]
        ### BEGIN SOLUTION
        cluster_distance = euklidean_distance_2d(x, y, center_x, center_y)
        if cluster_distance < min_distance:
            min_distance = cluster_distance
            min_index = cluster_index
        ### END SOLUTION
    return min_index

def assign_clusters(data_x, data_y, cluster_indices, cluster_centers):
    cluster_assignments = []
    ### BEGIN SOLUTION
    for x, y in zip(data_x, data_y):
        cluster_index = assign_to_cluster(x, y, cluster_indices, cluster_centers)
        cluster_assignments.append(cluster_index)
    ### END SOLUTION
    return cluster_assignments

In [ ]:
_indices = [1, 2, 3]
cluster_centers = [(1, 1), (3.1, 3.1), (5, 5)]
assert assign_to_cluster(1, 1, _indices, cluster_centers) == 1
assert assign_to_cluster(2, 2, _indices, cluster_centers) == 1
assert assign_to_cluster(3, 3, _indices, cluster_centers) == 2
assert assign_to_cluster(100, 100, _indices, cluster_centers) == 3
assert assign_to_cluster(100, 100, [3, 2, 1], cluster_centers) == 1
test_x = [1, 2, 3, 4, 5]
test_y = [1, 2, 3, 5, 6]
_assignments = assign_clusters(test_x, test_y, _indices, cluster_centers)
assert _assignments[0] == 1
assert _assignments[1] == 1
assert _assignments[2] == 2
assert _assignments[3] == 3
assert _assignments[4] == 3

#### 1 C) Calculation of the cluster centers (centers of gravity)

In this subtask, step 3) of the *K-Means* algorithm, i.e. the (re)calculation of the cluster centers, is considered.

To implement this step, the function ``compute_cluster_centers(data_x, data_y, cluster_indices, cluster_assignments)`` is to be implemented.
The parameters correspond to those described above (see previous subtask).
A list ``new_cluster_centers`` is to be returned, which returns the newly calculated cluster center of gravity (cluster center) for each cluster.

As a reminder: The center of gravity of a set of points $(x, y)_i \in \mathbb{R}^2$, $i=1,...,n$ can be calculated as $(\sum_{i=1}^n \frac{x_i}{n}, \sum_{i=1}^n \frac{y_i}{n})$.

Proceed as follows for the implementation:
1. for each cluster, first add the x- and y-coordinates of all data points assigned to this cluster to the lists ``cluster_xs`` and ``cluster_ys``.
2. calculate the x and y coordinates of the cluster centroid and store them in the variables ``center_x`` and ``center_y``.

In [ ]:
def compute_cluster_centers(data_x, data_y, cluster_indices, cluster_assignments):
    new_cluster_centers = []
    for c in cluster_indices:
        cluster_xs = []
        cluster_ys = []
        ### 1. Add x- and y- coordinates of all the cluster's data points to cluster_xs and cluster_ys, respectively.
        ### BEGIN SOLUTION
        for i, (x, y) in enumerate(zip(data_x, data_y)):
            if cluster_assignments[i] == c:
                cluster_xs.append(x)
                cluster_ys.append(y)
        ### END SOLUTION
        ### 2. Compute cluster center's x- and y- coordinates as center_x and center_y
        ### BEGIN SOLUTION
        center_x = sum(cluster_xs) / len(cluster_xs)
        center_y = sum(cluster_ys) / len(cluster_ys)
        ### END SOLUTION
        new_cluster_centers.append((center_x, center_y))
    return new_cluster_centers

In [ ]:
test_x = [1, 2, 3, 4, 5]
test_y = [1, 2, 3, 5, 6]
_indices = [1, 2, 3]
_assignments = [1, 2, 3, 1, 2]
_centers = compute_cluster_centers(test_x, test_y, _indices, _assignments)
assert len(_centers) == len(_indices)
assert _centers[0][0] == 5/2 and _centers[0][1] == 6/2
assert _centers[1][0] == 7/2 and _centers[1][1] == 8/2
assert _centers[2][0] == 3 and _centers[2][1] == 3

#### 1 D) K-Means algorithm

The implemented sub-steps can now be combined into the iterative procedure for cluster (re)determination of the *K-Means* algorithm.

The function ``k_means_clustering(data_x, data_y, cluster_indices, cluster_centers, iteration_number)`` is implemented for this purpose.
The parameters ``data_x, data_y, cluster_indices, cluster_centers`` contain the data described above.
The additional parameter ``iteration_number`` specifies how many iterations of the cluster improvement should be carried out before the loop is aborted.

The algorithm should proceed as described in the lecture (see figure above).
Specifically, for ``i`` iterations:
* Step 3) (calculation of the cluster centroids) should be executed exactly ``i`` times
* Step 2) (cluster assignment) is to be carried out exactly ``i+1`` times

The function returns both the list of cluster centers ``cluster_centers`` and the list of assignments of all data points to the cluster indices ``cluster_assignments``.

In [ ]:
def k_means_clustering(data_x, data_y, cluster_indices, cluster_centers, iteration_number):
    ### BEGIN SOLUTION
    for i in range(iteration_number):
        cluster_assignments = assign_clusters(data_x, data_y, cluster_indices, cluster_centers)
        cluster_centers = compute_cluster_centers(data_x, data_y, cluster_indices, cluster_assignments)

    cluster_assignments = assign_clusters(data_x, data_y, cluster_indices, cluster_centers)
    
    return cluster_centers, cluster_assignments
    ### END SOLUTION

In [ ]:
cluster_indices = [1, 2, 3]
cluster_centers = [(1, 1), (3.1, 3.1), (5, 5)]
test_x = [1, 2, 3, 4, 5]
test_y = [1, 2, 3, 5, 6]
_centers, _assignments = k_means_clustering(test_x, test_y, cluster_indices, cluster_centers, 0)
assert _centers[0][0] == cluster_centers[0][0] and _centers[0][1] == cluster_centers[0][1]
assert _centers[1][0] == cluster_centers[1][0] and _centers[1][1] == cluster_centers[1][1]
assert _centers[2][0] == cluster_centers[2][0] and _centers[2][1] == cluster_centers[2][1]
assert _assignments[0] == 1
assert _assignments[1] == 1
assert _assignments[2] == 2
assert _assignments[3] == 3
assert _assignments[4] == 3
_centers, _assignments = k_means_clustering(test_x, test_y, cluster_indices, cluster_centers, 1)
assert _centers[0][0] == _centers[0][1] == 1.5
assert _centers[1][0] == _centers[1][1] == 3
assert _centers[2][0] == 4.5
assert _centers[2][1] == 5.5

#### 1 E) Random initialization

In order to execute the implemented algorithm, an initialization of the clusters is required.
This includes both the number of clusters and the choice of initial cluster centers.

To do this, we implement a randomized selection of cluster centers from the set of all data points in the function ``pick_random_cluster_centers(data_x, data_y, cluster_number)``.
Complete the function so that the list of cluster centers ``cluster_centers`` is initialized and filled,
so that it contains the data points with the randomly selected indices from ``initial_center_indices``.

Then use the functions completed so far to combine the complete *K-Means* algorithm with random initialization of the cluster centers in the function ``k_means_random_init(data, cluster_number, iteration_number)``.
In addition to the data set (``data``), the arguments of the function contain the number of clusters into which the data is to be divided and the number of iterations to be performed.

Within the function, therefore:
* x and y columns must be extracted from the total data set ``data``; ``data`` is an array with dimension ``(n, 2)``, where ``n`` corresponds to the number of data points in the data set
* the indices of the clusters (e.g. ``[1, 2, ..., cluster_number]``) are selected and written to the ``cluster_indices`` list
* an initialization of the cluster centers is found using the function ``pick_random_cluster_centers`` and stored in the variable ``cluster_centers``

The returns of the function are, as above, the list of cluster centers ``cluster_centers`` and the list of the assignments of all data points to the cluster indices ``cluster_assignments``.

In [ ]:
import random

def pick_random_cluster_centers(data_x, data_y, cluster_number):
    data_indices = list(range(len(data_x)))
    random.shuffle(data_indices)
    initial_center_indices = data_indices[:cluster_number]
    ### BEGIN SOLUTION
    cluster_centers = [(data_x[index], data_y[index]) for index in initial_center_indices]
    ### END SOLUTION
    return cluster_centers

def k_means_random_init(data, cluster_number, iteration_number):
    ### BEGIN SOLUTION
    data_x = data[:, 0]
    data_y = data[:, 1]
    cluster_indices = list(range(cluster_number))
    cluster_centers = pick_random_cluster_centers(data_x, data_y, cluster_number)
    ### END SOLUTION
    return k_means_clustering(data_x, data_y, cluster_indices, cluster_centers, iteration_number)

In [ ]:
random.seed(10)
test_x = [1, 2, 3, 4, 5]
test_y = [1, 2, 3, 5, 6]
_centers = pick_random_cluster_centers(test_x, test_y, 2)
assert _centers[0][0] == 4
assert _centers[0][1] == 5
assert _centers[1][0] == 3
assert _centers[1][1] == 3
_centers, _assignments = k_means_random_init(df.values, 2, 2)

#### 1 F) Application and visualization

Now the fully implemented *K-Means* algorithm is to be applied to the data set read in at the beginning and the results visualized.

To do this, the function ``k_means_random_init`` must be called. 
Select the appropriate number of clusters for the data set and perform 10 iterations of the algorithm.
Save the clustering results in the variables ``cluster_centers`` and ``cluster_assignment``.

The clustering results should then be visualized using the ``plot_df_scatter`` function.

**Attention:**
If the previous parts of the task have not been solved (correctly), the next code cell can be executed to replace the clustering algorithm with the variant already implemented in the *sklearn* package.

Execute the algorithm with random cluster initialization several times.
What do you notice?

In [ ]:
### OPTIONAL: ONLY RUN THIS CODE CELL IF YOU DID NOT SUCCESSFULLY IMPLEMENT THE K-MEANS ALGORITHM YOURSELF
from sklearn import cluster

def k_means_random_init(data, cluster_number, iteration_number):
    kmeans_model = cluster.KMeans(n_clusters=cluster_number, n_init=1, init="random")
    kmeans_model.n_iter_ = iteration_number
    cluster_predictions = kmeans_model.fit_predict(data)

    return kmeans_model.cluster_centers_, cluster_predictions

In [ ]:
### BEGIN SOLUTION
cluster_centers, cluster_assignment = k_means_random_init(df.values, 4, 10)
plot_df_scatter(df, "Power", "Speed", cluster_assignment)
### END SOLUTION

In [ ]:
assert len(cluster_centers) == 4
assert len(cluster_assignment) == len(df.values)
for center in cluster_centers:
    assert len(center) == 2

#### Bonus: Intelligent initialization

In this task, you can come up with a "smarter" initialization idea (heuristic) and implement it yourself.

The aim is, of course, to find a method where good clustering results are not dependent on a good random seed.

To do this, implement the method ``initialize_clusters_clever(data, n_clusters, random_state)``. The parameters of the method are:
* ``data``: The data to be clustered in an array of dimension ``(n, 2)``.
* ``n_clusters``: The number of clusters to be optimized
* ``random_state``: A specific object to initialize the random factor to ensure reproducibility (can be ignored, especially for deterministic implementations)

As before, a list of the selected cluster centers (tuples) with length ``n_clusters`` or an array of dimension ``(n_clusters, 2)`` should be returned.

Of course, you should find an implementation that is as generally valid as possible, i.e. you should not work with absolute numbers that are tuned to the existing data set!

Then test the implementation by running the K-Means algorithm with the new initialization and analyzing the results manually (in the plot).

*Hint*: A commonly used initialization strategy is K-means++. Here, the first cluster center is chosen randomly from the data points. Succeedingly, cluster centers are selected from the remaining data points. Thereby, the probability of choosing one of the data points as the next center is proportional to the summed distance of this point to the already selected cluster centers.

In [ ]:
def initialize_clusters_clever(data, n_clusters, random_state):
    random.seed(hash(random_state))
    cluster_centers = [(0, 0) for index in range(n_clusters)]

    ### BEGIN SOLUTION
    # Select the first cluster center randomly
    cluster_center_indices = [random.randint(0, data.shape[0])]
    
    while len(cluster_center_indices) < n_clusters:

        # Compute the sum of distances to cluster centers for all data points (which are not yet cluster centers)
        point_distances = []
        for i in range(len(data)):
            point_cluster_distance_sum = 0
            if i not in cluster_center_indices:
                for j in cluster_center_indices:
                    point_cluster_distance_sum += euklidean_distance_2d(data[i,0], data[i,1], data[j,0], data[j,1])
            point_distances.append(point_cluster_distance_sum)

        # Compute probability distribution for next cluster center choice proportional to distances
        all_distances_sum = sum(point_distances)
        point_probs = [dist / sum(point_distances) for dist in point_distances]

        # Choose the next cluster center according to probability distribution
        random_number = random.random()
        prob_sum = 0
        i = 0
        while random_number >= prob_sum:
            prob_sum += point_probs[i]
            i += 1

        cluster_center_indices.append(i-1)
    
    cluster_centers = [data[i] for i in cluster_center_indices]
    ### END SOLUTION
    return cluster_centers

In [ ]:
def k_means_clever_init(data, cluster_number, iteration_number):
    kmeans_model = cluster.KMeans(n_clusters=cluster_number, n_init=1, init=initialize_clusters_clever)
    kmeans_model.n_iter_ = iteration_number
    cluster_predictions = kmeans_model.fit_predict(data)

    return kmeans_model.cluster_centers_, cluster_predictions

cluster_centers, cluster_assignment = k_means_clever_init(10 * df.values, 4, 10)
plot_df_scatter(df * 10, "Power", "Speed", cluster_assignment)